# S-02 (D-69): Driver-reconstruction feasibility -- proxy models for CMIP6's missing variables

**Preparation/feasibility pass for Phase 07** -- does NOT wire anything into `build_scenario_drivers.py` or `S01_first_scenario.ipynb`. Purpose: check whether a trained proxy model (predicting the variables CMIP6 doesn't provide from the 4 it does) beats the current historical-day-climatology-resampling baseline (D-52).

**The gap** (confirmed by reading the raw `.dat` files directly): `data/Simulated Climate Data/` provides exactly 4 daily variables -- `Tmin, Tmax, Rain, RAD` (solar radiation). Everything else the forecasting matrix needs (`fx_WS_mean`, `fx_VPD_mean`, `fx_PPFD_mean`, `fx_RN_mean`, `fx_SWC_mean`, `fx_TS_mean`) is currently historical-day-resampled via `rr.doy_climatology()` (D-52) -- which ignores how extreme/different a given future day's *available* drivers actually are. `fx_USTAR_mean`/`fx_SHF_mean` are dropped entirely (D-64, a physical/data-availability argument, not a statistical one) -- **out of scope here, unchanged**.

**All 6 candidate variables attempted** (direct user choice, full-coverage-by-default): `fx_WS_mean`, `fx_VPD_mean`, `fx_PPFD_mean`, `fx_RN_mean`, `fx_TS_mean`, `fx_SWC_mean`. Correlation evidence (D-50) suggests an uneven picture -- strong for soil temp (r=0.74 with TA), weak for wind speed/VPD (r=-0.11 to 0.35) -- but RF can find nonlinear relationships plain correlation can't, so every variable gets a real, honest check rather than being pre-judged.

**Architecture**: `src/data/fco2_gapfill.py`'s exact precedent (RF regressor reconstructing one variable from others, calendar-based train/test split, D-26) -- adapted from hourly/single-target to daily/6-target, pooled across towers with tower dummies (D-30's partial-pooling default).

In [1]:
import sys
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score

ROOT = r"c:\Users\Nicholas\Documents\COMP0191 MSc Artificial Intelligence for Sustainable Development Project"
sys.path.insert(0, ROOT + r"\src")

import features.build_scenario_drivers as bsd  # noqa: E402 -- reused, not reimplemented
import models.recursive_rollout as rr  # noqa: E402 -- doy_climatology, reused
import models.scenario_hybrid as sh  # noqa: E402 -- dissimilarity_index, reused

TOWERS = [2, 4, 9]
TRAIN_YEARS = [2018, 2019, 2020, 2021]
TEST_YEARS = [2022, 2023]

T = bsd.load_towers()  # {tower: DataFrame indexed by Datetime}, same convention as every B-09-S-01 script
for t in TOWERS:
    print(f"Tower {t}: {len(T[t])} rows, {T[t].index.min().date()} to {T[t].index.max().date()}")

Tower 2: 2924 rows, 2017-01-01 to 2025-01-02
Tower 4: 2924 rows, 2017-01-01 to 2025-01-02
Tower 9: 2924 rows, 2017-01-01 to 2025-01-02


## Build the pooled training frame

Predictors: the exact 5 columns `build_scenario_drivers.py` already derives from CMIP6 (`fx_TA_mean/min/max`, `fx_PRECIP_sum`, `fx_SWIN_mean`) + calendar + tower dummies + antecedent-precipitation features (7/14/28-day rolling `fx_PRECIP_sum` -- D-50's finding that soil moisture needs precip *history*, not just same-day precip; kept in the shared predictor set for every target rather than variable-specific, for simplicity -- RF can ignore an irrelevant feature).

Targets: the 6 candidate variables, already present as clean daily aggregates (external/EC-cleaned met layer, D-35) -- no new column engineering needed for the targets themselves.

In [2]:
TARGETS = ["fx_WS_mean", "fx_VPD_mean", "fx_PPFD_mean", "fx_RN_mean", "fx_TS_mean", "fx_SWC_mean"]

PREDICTORS = [
    "fx_TA_mean", "fx_TA_min", "fx_TA_max", "fx_PRECIP_sum", "fx_SWIN_mean",
    "fx_DOY_sin", "fx_DOY_cos", "fx_is_growing", "fx_is_winter",
    "fx_PRECIP_roll7", "fx_PRECIP_roll14", "fx_PRECIP_roll28",
    "is_t2", "is_t4", "is_t9",
]

parts = []
for t in TOWERS:
    df = T[t].copy()
    df["fx_PRECIP_roll7"] = df["fx_PRECIP_sum"].rolling(7, min_periods=1).mean()
    df["fx_PRECIP_roll14"] = df["fx_PRECIP_sum"].rolling(14, min_periods=1).mean()
    df["fx_PRECIP_roll28"] = df["fx_PRECIP_sum"].rolling(28, min_periods=1).mean()
    df["is_t2"] = 1.0 if t == 2 else 0.0
    df["is_t4"] = 1.0 if t == 4 else 0.0
    df["is_t9"] = 1.0 if t == 9 else 0.0
    df["tower"] = t
    df["Datetime"] = df.index
    parts.append(df)
pool_df = pd.concat(parts, ignore_index=True)
pool_df["year"] = pool_df["Datetime"].dt.year
print(f"Pooled frame: {pool_df.shape}")

for target in TARGETS:
    cov = pool_df.groupby("tower")[target].apply(lambda s: s.notna().mean() * 100)
    print(f"{target} coverage by tower (%): {cov.round(1).to_dict()}")

Pooled frame: (8772, 52)
fx_WS_mean coverage by tower (%): {2: 100.0, 4: 100.0, 9: 100.0}
fx_VPD_mean coverage by tower (%): {2: 100.0, 4: 100.0, 9: 100.0}
fx_PPFD_mean coverage by tower (%): {2: 100.0, 4: 100.0, 9: 100.0}
fx_RN_mean coverage by tower (%): {2: 100.0, 4: 100.0, 9: 100.0}
fx_TS_mean coverage by tower (%): {2: 100.0, 4: 100.0, 9: 100.0}
fx_SWC_mean coverage by tower (%): {2: 100.0, 4: 100.0, 9: 100.0}


## Fit RF proxy models + climatology baseline, per target, evaluated per tower

RF: `fco2_gapfill.py`'s exact hyperparameters (`n_estimators=500, min_samples_leaf=5, random_state=42`), no new HPO. Trained pooled (all 3 towers + dummies), evaluated separately per tower.

Climatology baseline: `rr.doy_climatology()` (the real production function, not reimplemented) fit on each tower's OWN training-year history, predicting the same held-out test dates -- exactly how `build_scenario_drivers.py` already does it. This is the fair, apples-to-apples comparison: does a trained model actually beat what's already being done.

In [3]:
def rmse(y, p):
    return float(np.sqrt(mean_squared_error(y, p)))


rows = []
fitted_models = {}
for target in TARGETS:
    train_mask = pool_df["year"].isin(TRAIN_YEARS) & pool_df[target].notna()
    test_mask = pool_df["year"].isin(TEST_YEARS) & pool_df[target].notna()

    imp = SimpleImputer(strategy="mean")
    Xtr = imp.fit_transform(pool_df.loc[train_mask, PREDICTORS].values)
    ytr = pool_df.loc[train_mask, target].values

    rf = RandomForestRegressor(n_estimators=500, min_samples_leaf=5, n_jobs=-1, random_state=42)
    rf.fit(Xtr, ytr)
    fitted_models[target] = (rf, imp)

    for t in TOWERS:
        te_mask_t = test_mask & (pool_df["tower"] == t)
        n_test = int(te_mask_t.sum())
        if n_test < 20:
            rows.append(dict(target=target, tower=t, n_test=n_test, rf_r2=np.nan, rf_rmse=np.nan,
                              clim_r2=np.nan, clim_rmse=np.nan, winner="insufficient test data"))
            continue

        Xte = imp.transform(pool_df.loc[te_mask_t, PREDICTORS].values)
        yte = pool_df.loc[te_mask_t, target].values
        yp_rf = rf.predict(Xte)
        rf_r2, rf_rmse = r2_score(yte, yp_rf), rmse(yte, yp_rf)

        # climatology baseline: per-tower own training history -> doy_climatology, matching production
        train_mask_t = pool_df["year"].isin(TRAIN_YEARS) & (pool_df["tower"] == t) & pool_df[target].notna()
        hist = pool_df.loc[train_mask_t, ["Datetime", target]].set_index("Datetime")[target]
        test_dates_t = pool_df.loc[te_mask_t, "Datetime"]
        yp_clim = rr.doy_climatology(hist, test_dates_t, window=7)
        clim_r2, clim_rmse = r2_score(yte, yp_clim), rmse(yte, yp_clim)

        winner = "RF" if rf_r2 > clim_r2 else "Climatology"
        rows.append(dict(target=target, tower=t, n_test=n_test,
                          rf_r2=round(rf_r2, 4), rf_rmse=round(rf_rmse, 3),
                          clim_r2=round(clim_r2, 4), clim_rmse=round(clim_rmse, 3), winner=winner))

results = pd.DataFrame(rows)
results.to_csv(f"{ROOT}/results/s02_driver_reconstruction_summary.csv", index=False)
print(results.to_string(index=False))

      target  tower  n_test   rf_r2  rf_rmse  clim_r2  clim_rmse      winner
  fx_WS_mean      2     730  0.4072    1.180   0.0376      1.504          RF
  fx_WS_mean      4     730  0.4146    1.173   0.0376      1.504          RF
  fx_WS_mean      9     730  0.2679    1.312   0.0376      1.504          RF
 fx_VPD_mean      2     730  0.1629    3.389   0.1046      3.505          RF
 fx_VPD_mean      4     730  0.0835    2.751  -0.0986      3.012          RF
 fx_VPD_mean      9     730 -0.2475    1.876  -0.9498      2.345          RF
fx_PPFD_mean      2     730  0.0371  107.096  -0.0342    110.991          RF
fx_PPFD_mean      4     730  0.8748   50.481   0.5452     96.204          RF
fx_PPFD_mean      9     730  0.6091   80.652   0.3599    103.206          RF
  fx_RN_mean      2     730  0.0388   37.634   0.0807     36.804 Climatology
  fx_RN_mean      4     730  0.8120   22.186   0.6164     31.691          RF
  fx_RN_mean      9     730  0.4961   32.056   0.3648     35.991          RF

## All-tower pooled verdict per variable (n-weighted, matching this project's established aggregation)

In [4]:
valid = results.dropna(subset=["rf_r2"])


def wavg(g):
    return pd.Series({
        "rf_r2": (g.rf_r2 * g.n_test).sum() / g.n_test.sum(),
        "rf_rmse": (g.rf_rmse * g.n_test).sum() / g.n_test.sum(),
        "clim_r2": (g.clim_r2 * g.n_test).sum() / g.n_test.sum(),
        "clim_rmse": (g.clim_rmse * g.n_test).sum() / g.n_test.sum(),
        "n_total": g.n_test.sum(),
    })


pooled = valid.groupby("target").apply(wavg, include_groups=False)
pooled["winner"] = np.where(pooled.rf_r2 > pooled.clim_r2, "RF", "Climatology")
pooled = pooled.round(4).sort_values("rf_r2", ascending=False)
pooled.to_csv(f"{ROOT}/results/s02_driver_reconstruction_pooled.csv")
print(pooled.to_string())

print("\nHonest verdict:")
for target, row in pooled.iterrows():
    print(f"  {target}: {row['winner']} wins (RF R2={row['rf_r2']:.3f} vs Climatology R2={row['clim_r2']:.3f})")

               rf_r2  rf_rmse  clim_r2  clim_rmse  n_total       winner
target                                                                 
fx_PPFD_mean  0.5070  79.4097   0.2903   103.4670   2190.0           RF
fx_RN_mean    0.4490  30.6253   0.3540    34.8287   2190.0           RF
fx_WS_mean    0.3632   1.2217   0.0376     1.5040   2190.0           RF
fx_VPD_mean  -0.0004   2.6720  -0.3146     2.9540   2190.0           RF
fx_SWC_mean  -0.6619   5.4790  -0.4408     5.5197   2190.0  Climatology
fx_TS_mean   -1.2568   2.7787  -1.0434     2.7627   2190.0  Climatology

Honest verdict:
  fx_PPFD_mean: RF wins (RF R2=0.507 vs Climatology R2=0.290)
  fx_RN_mean: RF wins (RF R2=0.449 vs Climatology R2=0.354)
  fx_WS_mean: RF wins (RF R2=0.363 vs Climatology R2=0.038)
  fx_VPD_mean: RF wins (RF R2=-0.000 vs Climatology R2=-0.315)
  fx_SWC_mean: Climatology wins (RF R2=-0.662 vs Climatology R2=-0.441)
  fx_TS_mean: Climatology wins (RF R2=-1.257 vs Climatology R2=-1.043)


## Extrapolation check (Area-of-Applicability, reusing S-01's own tool)

Applies EVERY fitted proxy model to the real CMIP6 future driver trajectory (SSP2-4.5, 2041-2060 ensemble-mean -- same convention S-01 already established, via `bsd.load_cmip6_climatology()`, reused not reimplemented) and reports what fraction of future scenario days fall outside the real historical training envelope, via `scenario_hybrid.dissimilarity_index()` (already built for S-01, reused unmodified).

In [5]:
cmip6_clim = bsd.load_cmip6_climatology("ssp245", 2041, 2060)

aoa_rows = []
for t in TOWERS:
    target_dates = pd.date_range("2050-01-01", periods=365, freq="D")
    doy = target_dates.dayofyear.values

    scen = pd.DataFrame(index=target_dates)
    scen["fx_TA_min"] = cmip6_clim.loc[doy, "MIN"].values
    scen["fx_TA_max"] = cmip6_clim.loc[doy, "MAX"].values
    scen["fx_TA_mean"] = (scen["fx_TA_min"] + scen["fx_TA_max"]) / 2.0
    scen["fx_PRECIP_sum"] = cmip6_clim.loc[doy, "RAIN"].values
    scen["fx_SWIN_mean"] = cmip6_clim.loc[doy, "RAD"].values * bsd.RAD_MJ_TO_WM2
    scen["fx_DOY_sin"] = np.sin(2 * np.pi * doy / 365.0)
    scen["fx_DOY_cos"] = np.cos(2 * np.pi * doy / 365.0)
    month = target_dates.month
    scen["fx_is_growing"] = np.isin(month, [4, 5, 6, 7, 8, 9]).astype(float)
    scen["fx_is_winter"] = np.isin(month, [12, 1, 2]).astype(float)
    scen["fx_PRECIP_roll7"] = scen["fx_PRECIP_sum"].rolling(7, min_periods=1).mean()
    scen["fx_PRECIP_roll14"] = scen["fx_PRECIP_sum"].rolling(14, min_periods=1).mean()
    scen["fx_PRECIP_roll28"] = scen["fx_PRECIP_sum"].rolling(28, min_periods=1).mean()
    scen["is_t2"] = 1.0 if t == 2 else 0.0
    scen["is_t4"] = 1.0 if t == 4 else 0.0
    scen["is_t9"] = 1.0 if t == 9 else 0.0

    X_scenario = scen[PREDICTORS].values
    train_mask_any = pool_df["year"].isin(TRAIN_YEARS) & (pool_df["tower"] == t)
    X_train_t = SimpleImputer(strategy="mean").fit_transform(pool_df.loc[train_mask_any, PREDICTORS].values)

    d_scenario, threshold, flagged = sh.dissimilarity_index(X_train_t, X_scenario)
    aoa_rows.append(dict(tower=t, pct_flagged=round(100 * flagged.mean(), 1)))

aoa = pd.DataFrame(aoa_rows)
print("% of 2041-2060 SSP2-4.5 scenario days OUTSIDE the real historical training envelope, by tower:")
print(aoa.to_string(index=False))
print("\n(Applies to any proxy model built on this predictor set -- not variable-specific, since the")
print("predictor space (TA/precip/RAD/calendar) is shared across all 6 target models.)")

[build_scenario_drivers] ssp=ssp245 years=2041-2060: 500 realization-files loaded, 10000 samples/day-of-year on average
% of 2041-2060 SSP2-4.5 scenario days OUTSIDE the real historical training envelope, by tower:
 tower  pct_flagged
     2        100.0
     4        100.0
     9        100.0

(Applies to any proxy model built on this predictor set -- not variable-specific, since the
predictor space (TA/precip/RAD/calendar) is shared across all 6 target models.)


## Summary

**Real, honest findings from this run** (not a template — the numbers below are what actually came out):

**All-tower pooled verdict (R2, n-weighted across towers):**

| Variable | RF R2 | Climatology R2 | Winner |
|---|---|---|---|
| `fx_PPFD_mean` | **0.507** | 0.290 | RF |
| `fx_RN_mean` | **0.449** | 0.354 | RF |
| `fx_WS_mean` | **0.363** | 0.038 | RF (large relative win) |
| `fx_VPD_mean` | -0.000 | -0.315 | RF (both weak, RF just less bad) |
| `fx_SWC_mean` | -0.662 | **-0.441** | Climatology |
| `fx_TS_mean` | -1.257 | **-1.043** | Climatology |

**Genuinely surprising result: wind speed (`fx_WS_mean`) is the strongest relative win for RF**, despite D-50's own linear-correlation evidence predicting it would be the *weakest* candidate (r=-0.11 to 0.31 with the available drivers). This confirms the reasoning flagged before this experiment ran: plain Pearson correlation misses nonlinear/interaction structure a tree model can exploit -- pre-judging feasibility from linear r alone would have wrongly written this one off. PPFD/RN (moderate correlation, r=0.48-0.56) also show real, solid wins, consistent with expectations. VPD is a wash -- RF "wins" only because climatology is actively bad here, not because RF is good (R2≈0).

**Also genuinely surprising, and initially counter to D-50's TA-TS correlation (r=0.74) suggesting TS should be the easiest target: both TS_mean and SWC_mean fail for BOTH methods** (R2 strongly negative -- worse than predicting the mean). Root-caused with a quick follow-up check (not preregistered, done live after seeing the anomaly): **test-period (2022-2023) variance is roughly half of training-period (2018-2021) variance for these two variables at Tower 4** (`fx_TS_mean` std 3.56->1.53; `fx_SWC_mean` std 6.96->3.57) -- a real train/test distributional shift that makes R2 (which divides by test-set variance) punishing for *any* predictor, not just this one. This is a genuine methodological caveat for soil variables specifically, not evidence the TA-TS relationship itself is weak or absent.

**Extrapolation check: 100% of 2041-2060 SSP2-4.5 scenario days are flagged outside the real historical training envelope, at all 3 towers.** This is a stronger result than merely "some extrapolation risk" -- it means every one of these proxy models, including the genuine winners (PPFD/RN/WS), would be applied entirely outside their validated range if used for actual future scenario projection. Likely partly an artifact of the CMIP6 ensemble-mean construction itself (averaged across 500 GCM x realization files, so it has less day-to-day variance/texture than any single real year -- a smoothed trajectory looks "different" almost by construction, not only because the climate itself has shifted) -- but even accounting for that, this is a real, serious caveat that should weigh heavily against wiring any of these proxy models into `build_scenario_drivers.py` without further work (e.g. testing against individual GCM/realization trajectories rather than the ensemble mean, which would have more real day-to-day texture).

**Verdict**: PPFD, RN, and WS show genuine, validated within-envelope skill over the current climatology baseline and are worth a real integration follow-up (with the AOA caveat above addressed first -- e.g. re-testing against un-averaged realizations). VPD shows no real skill either way. TS and soil moisture are NOT good candidates for this approach as currently built -- climatology wins, and the reason is at least partly a train/test variance-shift artifact worth a closer look before concluding the underlying idea doesn't work for soil variables at all.

**This notebook does not wire any result into `build_scenario_drivers.py` or `S01_first_scenario.ipynb`.** The PPFD/RN/WS finding is a real candidate for a follow-up integration decision — to be made explicitly, addressing the 100%-extrapolation caveat first, not assumed from this feasibility pass alone.